# 01 - Dense Baseline

Plan section 104: the dense baseline trains, resumes, and reproduces its metrics from seed.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Dense baseline (plan section 104)

Proves the first notebook's three claims: the model trains,
checkpoint resume is parameter-for-parameter, and metrics are
reproducible from a fixed seed. Uses the tiny deterministic gate
model and the shared runtime - not the declared 285M config.


In [ ]:
import torch
from act.model import DenseBaselineConfig, DenseCausalLM
from act.training.runtime import (
    TrainingConfig,
    TrainingState,
    load_checkpoint,
    make_optimizer,
    save_checkpoint,
    train_steps,
)

CHECKPOINT = 'artifacts/dense-baseline.pt'
STEPS = 8

torch.manual_seed(19)
model = DenseCausalLM(DenseBaselineConfig())
optimizer = make_optimizer(model, TrainingConfig())
state = TrainingState()
losses = train_steps(model, optimizer, state, TrainingConfig(), STEPS)
print('steps', state.step, '| tokens', state.tokens,
      '| final loss', round(losses[-1], 6))

In [ ]:
# Claim 2: a checkpoint resumes parameter-for-parameter.
save_checkpoint(CHECKPOINT, model, optimizer, state)
resumed_model = DenseCausalLM(DenseBaselineConfig())
resumed_optimizer = make_optimizer(resumed_model, TrainingConfig())
resumed_state = load_checkpoint(CHECKPOINT, resumed_model, resumed_optimizer)
same = all(
    torch.equal(a.cpu(), b.cpu())
    for a, b in zip(model.state_dict().values(), resumed_model.state_dict().values())
)
assert same, 'resume did not reproduce parameters exactly'
assert resumed_state.step == state.step
print('resume reproduced', len(resumed_model.state_dict()),
      'tensors at step', resumed_state.step)

In [ ]:
# Claim 3: a fresh run from the same seed is metric-identical.
torch.manual_seed(19)
fresh_model = DenseCausalLM(DenseBaselineConfig())
fresh_optimizer = make_optimizer(fresh_model, TrainingConfig())
fresh_losses = train_steps(
    fresh_model, fresh_optimizer, TrainingState(), TrainingConfig(), STEPS
)
assert fresh_losses == losses, 'training is not seed-reproducible'
print('fresh run reproduced every loss exactly')

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '01_dense_baseline', 'steps': STEPS,
 'final_loss': losses[-1], 'tokens': state.tokens,
 'checkpoint': CHECKPOINT,
 'resume_reproduced': same, 'seed_reproducible': True}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '01-dense-baseline.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)